# Evaluating the Ideal Chunk Size for a RAG System using LlamaIndex

> **Refreshed November 2026** — modernized for the modular `llama-index-core` 0.14
> architecture (`Settings` instead of `ServiceContext`, `llama_index.core.evaluation`)
> and the GPT-6 generation of models.

# **Introduction**

Retrieval-augmented generation (RAG) fuses the extensive retrieval capabilities of search systems with the LLM. When implementing a RAG system, one critical parameter that governs the system's efficiency and performance is the `chunk_size`. How does one discern the optimal chunk size for seamless retrieval? This is where LlamaIndex **Response Evaluation** comes in handy. In this notebook, we'll guide you through the steps to determine the best `chunk_size` using LlamaIndex's response evaluation module. If you're unfamiliar with it, review the [evaluation module guide](https://docs.llamaindex.ai/en/stable/module_guides/evaluating/) before proceeding.

## **Why Chunk Size Matters**

Choosing the right `chunk_size` influences the efficiency and accuracy of a RAG system in several ways:

1. **Relevance and Granularity**: A small `chunk_size`, like 128, yields more granular chunks. This granularity, however, presents a risk: vital information might not be among the top retrieved chunks, especially if the `similarity_top_k` setting is as restrictive as 2. Conversely, a chunk size of 512 is likely to encompass all necessary information within the top chunks. To navigate this, we employ the **Faithfulness** and **Relevancy** metrics, which measure the absence of 'hallucinations' and the relevancy of responses with respect to the query and retrieved contexts.
2. **Response Generation Time**: As the `chunk_size` increases, so does the volume of information directed into the LLM. While this ensures more comprehensive context, it can also slow the system down.

In essence, determining the optimal `chunk_size` is about striking a balance: capturing all essential information without sacrificing speed.

## **Setup**

All dependencies are listed in `requirements.txt` (`llama-index`, `pypdf`, `python-dotenv`, `nest-asyncio`). Install them with `pip install -r requirements.txt` before running this notebook.

In [1]:
import nest_asyncio

nest_asyncio.apply()

import os
import time

from dotenv import load_dotenv

# Modern (0.10+) import paths — everything lives under llama_index.core
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.evaluation import (
    DatasetGenerator,
    FaithfulnessEvaluator,
    RelevancyEvaluator,
)
from llama_index.llms.openai import OpenAI

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY not found. Create a .env file in the project root "
        "with OPENAI_API_KEY=your_key_here"
    )

print("✅ Imports successful!")

✅ Imports successful!


## **Download Data**

We'll use the Uber 10-K SEC Filing for 2021 for this experiment.

In [ ]:
!mkdir -p 'data/10k/'
!wget -q 'https://raw.githubusercontent.com/jerryjliu/llama_index/main/docs/examples/data/10k/uber_2021.pdf' -O 'data/10k/uber_2021.pdf'
!ls -lh 'data/10k/uber_2021.pdf'

## **Load Data**

Let's load the document.

In [2]:
# Load Data
reader = SimpleDirectoryReader("./data/10k/")
documents = reader.load_data()
print(f"Loaded {len(documents)} document(s), {sum(len(d.text) for d in documents):,} chars")

Loaded 307 document(s), 1,267,377 chars


## **Question Generation**

To select the right `chunk_size`, we'll compute Average Response Time, Faithfulness, and Relevancy for various sizes. The `DatasetGenerator` generates questions from the documents. We use a cheap, fast model (`gpt-6-luna`) for question generation since volume matters more than quality here.

In [ ]:
# Generate 40 questions from the first 20 pages
eval_documents = documents[:20]

question_gen_llm = OpenAI(model="gpt-5-nano", temperature=0.3)

# Build nodes first, then construct the generator directly.
# (DatasetGenerator.from_documents internally applies a KeywordNodePostprocessor
# that requires an optional spacy install — constructing from nodes avoids it.)
question_gen_splitter = SentenceSplitter(chunk_size=512)
question_gen_nodes = question_gen_splitter.get_nodes_from_documents(eval_documents)

data_generator = DatasetGenerator(
    nodes=question_gen_nodes,
    llm=question_gen_llm,
)
eval_questions = data_generator.generate_questions_from_nodes(num=40)

print(f"✅ Generated {len(eval_questions)} evaluation questions")
print(f"   Example: {eval_questions[0]}")

/var/folders/cj/13vbmk7n7fqgmdnqjjwn1bx80000gn/T/ipykernel_11759/1132716926.py:12: DeprecationWarning: Call to deprecated class DatasetGenerator. (Deprecated in favor of `RagDatasetGenerator` which should be used instead.)
  data_generator = DatasetGenerator(
2026-11-15 19:16:44,268 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-15 19:16:44,334 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-15 19:16:44,513 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-15 19:16:46,363 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-15 19:16:46,406 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-15 19:16:46,537 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-15 19:16:46,599 - INFO - HTTP Request: POST https://ap

## Setting Up Evaluators

We use a strong model (`gpt-6.1-sol`) as the judge for evaluating responses:

1. **FaithfulnessEvaluator** — measures whether the response was hallucinated, i.e. whether it is supported by the retrieved source nodes.
2. **RelevancyEvaluator** — measures whether the response actually answers the query.

In [ ]:
# Strong model as the evaluation judge (temperature 0 for consistency)
eval_llm = OpenAI(model="gpt-5-nano", temperature=0)

faithfulness_evaluator = FaithfulnessEvaluator(llm=eval_llm)
relevancy_evaluator = RelevancyEvaluator(llm=eval_llm)

print("✅ Evaluators initialized (judge: gpt-6.1-sol)")

## **Response Evaluation For A Chunk Size**

We evaluate each `chunk_size` on 3 metrics: Average Response Time, Average Faithfulness, and Average Relevancy. The function below handles node construction, index building, querying, and metric calculation.

In [ ]:
def evaluate_response_time_and_accuracy(chunk_size, eval_questions):
    """Evaluate average response time, faithfulness, and relevancy for a chunk size.

    Responses are generated by gpt-5-nano and judged by gpt-5-nano.
    """
    total_response_time = 0
    total_faithfulness = 0
    total_relevancy = 0

    # Configure the response-generation model (the judge evaluators keep their own LLM)
    Settings.llm = OpenAI(model="gpt-6.1-terra", temperature=0.1)
    Settings.chunk_size = chunk_size

    # Build nodes with the target chunk size, then index them
    # Overlap must be smaller than chunk_size (20% of chunk, capped at 200)
    splitter = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=min(200, chunk_size // 5))
    nodes = splitter.get_nodes_from_documents(eval_documents)
    vector_index = VectorStoreIndex(nodes)

    # By default, similarity_top_k is 2 — pass it to as_query_engine() to experiment
    query_engine = vector_index.as_query_engine()

    num_questions = len(eval_questions)
    for question in eval_questions:
        start_time = time.time()
        response_vector = query_engine.query(question)
        elapsed_time = time.time() - start_time

        faithfulness_result = faithfulness_evaluator.evaluate_response(
            response=response_vector
        ).passing

        relevancy_result = relevancy_evaluator.evaluate_response(
            query=question, response=response_vector
        ).passing

        total_response_time += elapsed_time
        total_faithfulness += faithfulness_result
        total_relevancy += relevancy_result

    average_response_time = total_response_time / num_questions
    average_faithfulness = total_faithfulness / num_questions
    average_relevancy = total_relevancy / num_questions

    return average_response_time, average_faithfulness, average_relevancy

## **Testing Across Different Chunk Sizes**

We evaluate a range of chunk sizes to find the best metrics. (For large question sets, wrap the evaluators in `BatchEvalRunner` from `llama_index.core.evaluation` to run evaluations concurrently.)

In [ ]:
# Iterate over chunk sizes; note this makes many LLM calls and can be slow/costly
for chunk_size in [128, 256, 512, 1024, 2048]:
    avg_response_time, avg_faithfulness, avg_relevancy = evaluate_response_time_and_accuracy(
        chunk_size, eval_questions
    )
    print(f"Chunk size {chunk_size} - Average Response time: {avg_response_time:.2f}s, "
          f"Average Faithfulness: {avg_faithfulness:.2f}, Average Relevancy: {avg_relevancy:.2f}")